## 1.1 Dimensiones de calidad con ejemplos concretos de la red SIATA

## Pregunta 1.1 — Qué significa que un dato hidrometeorológico tenga calidad

### (a) Calidad como aptitud para el uso, con dimensiones de un marco formal

Un registro hidrometeorológico tiene calidad cuando **sirve para la decisión que lo consume**: activar una alerta temprana en el Valle de Aburrá, calibrar un modelo lluvia-escorrentía o responder a un ciudadano. No es un atributo absoluto: el mismo minuto de datos puede ser apto para un boletín climático mensual e inútil para una alerta por creciente súbita. Uso esa definición de
*fitness for purpose* y la aterrizo en cuatro dimensiones de **ISO/IEC 25012**(coherentes con DAMA-DMBOK):

- **Completitud.** El piranómetro 6004 entrega 214.824 de los 217.440 minutos esperados entre el 1-dic-2025 y el 30-abr-2026 (98,80 %): 2.616 minutos de silencio que un modelo hidrológico leería como "no llovió" si no se declaran.

- **Exactitud.** El nivel 803 viene almacenado en coma flotante de precisión simple (p. ej. 6,0999755859375 m): la cuantización de ±0,05 m consume directamente el margen de error de los umbrales de desbordamiento.

- **Oportunidad.** El salto de 7,6 m a 20,5 m en un minuto en el nivel 803 (2026-03-16 00:03) solo tiene valor si se bandera y publica en minutos: el DAGRD decide en tiempo real, no al cierre del mes.

- **Consistencia.** En la pluviométrica 35, los canales p1 y p2 miden el mismo fenómeno y discrepan minuto a minuto (17:15 del 1-dic: p1 = 0,254 y p2 = 0,0): dos versiones internas contradictorias de la misma lluvia.

### (b) Calidad del dato vs. calidad del instrumento vs. calidad del proceso

- **Calidad del instrumento** es una propiedad física y documentada: exactitud,calibración y trazabilidad metrológica del sensor. Ejemplo: el balancín de la estación 35 con resolución de 0,254 mm por volcamiento; su certificado de   calibración con fecha de vencimiento; la deriva que pueda adquirir ellimnímetro 803 tras un mantenimiento.
- **Calidad del proceso** es cómo el dato se adquiere, valida, almacena y publica: reglas versionadas, idempotencia de reprocesos, metadatos y turnos  de operación. Los registros fusionados que aparecen en los crudos
  (`...0.0,0.254,151235,2025-12-01 18:10...` en la estación 35 y
  `1108.63,15836004,2026-04-21 19:22...` en la 6004) son fallas **del proceso**
  (serialización/formato), no del sensor.
- **Calidad del dato** es el resultado: el registro con sus banderas y sus
  metadatos. Las tres fallan de forma independiente: instrumento calibrado con
  proceso roto produce datos corruptos (las filas fusionadas); proceso
  impecable con instrumento descalibrado produce datos precisos pero sesgados
  (bias sistemático del 803 que ningún rango detecta); y un dato íntegro puede
  ser inútil para un uso concreto si llega tarde o sin metadatos (aptitud).

### (c) Por qué un dato físicamente plausible no es necesariamente correcto

La plausibilidad (estar dentro de rangos físicos) es condición necesaria, no
suficiente. Tres contraejemplos del propio entregable:

1. **Persistencia congelada.** El nivel 803 repite 18,10 m durante 14 minutos
   (2026-03-17 06:59–07:12). Cada valor es plausible; una quebrada no repite
   su nivel al milímetro durante 14 minutos: es sensor congelado. El rango no
   lo detecta; la prueba de persistencia sí.
2. **Inconsistencia interna.** En la estación 35, p1 = 0,254 con p2 = 0,0 en el
   mismo minuto: ambos valores plausibles por separado, mutuamente
   contradictorios en conjunto. Solo una prueba de coherencia entre columnas
   lo revela.
3. **Incoherencia contextual.** Un volcamiento aislado de 0,254 mm en madrugada
   seca, sin eco en ninguna otra estación de la cuenca, es una magnitud
   perfectamente plausible y un contexto sospechoso. La plausibilidad no
   evalúa contexto espacial ni temporal.

Por eso el control de calidad debe ser una **cadena de pruebas** (formato y completitud → rango → persistencia y salto → coherencia interna → coherencia espacial) y no un filtro de rangos.

## 1.2 Niveles de madurez del dato: crudo → validado → consumo (tabla con ejemplos SIATA)

| Aspecto | Dato crudo | Dato validado con metadatos | Dato para consumo del usuario |
|---|---|---|---|
| Definición | Registro tal como lo emite el datalogger, sin juicio: la fila `6004, 2026-04-30 20:22, 482.93` entra igual que una medición real | El mismo registro tras control de calidad con banderas por regla; el valor original se conserva, no se sobrescribe | Producto derivado listo para decidir: serie horaria de precipitación de la estación 35 que consumen el portal público y el modelo de alerta |
| Unidades y resolución | Como las graba el sensor: nivel 803 en m con flotante de precisión simple, radiación 6004 en W/m², marca de tiempo por minuto; sin homogeneizar (p1 y p2 de la pluviométrica 35 sin diccionario declarado) | Mismas unidades pero declaradas y documentadas (m, W/m², mm, °C, hPa, m/s, °); resolución nominal por minuto contrastada con la efectiva mediante conteo de huecos (2.616 en 6004) | Unidades homogeneizadas para el usuario (mm/h, mm/día, m, W/m²) y resolución regular horaria/diaria con agregación explícita: suma para precipitación, promedio para radiación, máximo para ráfaga (vv_max de 367) |
| Banderas de calidad | Ninguna, o solo la columna `calidad` original del sensor sin verificación independiente | Diccionario completo tipo OMM/WIGOS (válido, faltante, cuestionable, erróneo) aplicado regla por regla y versionado; el 482,93 W/m² nocturno queda como "erróneo" con regla y fecha | Banderas simplificadas: se publica solo el valor válido o el cuestionable con advertencia visible; el usuario recibe confianza por producto, no la columna de banderas interna |
| Metadatos mínimos | Nombre de archivo, código de estación (35, 367, 6004, 803) y fecha de descarga | Identificador WIGOS, coordenadas y altura del sensor, unidades, resolución, intervalo de muestreo (1 min), zona horaria America/Bogotá (UTC−5), versión del algoritmo de validación, diccionario de banderas, historial de calibración (ISO 19115 / WIGOS) | Todo lo anterior más licencia de uso, identificador persistente, fecha de actualización, limitaciones conocidas y cita recomendada del producto |
| Tratamiento de faltantes | Hueco como fila ausente: los 2.616 minutos de 6004 simplemente no están en el CSV | Hueco explícito: valor nulo más bandera "faltante"; jamás se rellena en silencio | Imputación documentada (interpolación climatológica o estación vecina de la misma cuenca) o NA visible con porcentaje de cobertura por período; método declarado en el producto |
| Usuario típico | Ingeniero de telemetría que depura el datalogger o la transmisión de la estación | Científico de datos o hidrólogo de SIATA que calibra modelos; auditor de calidad | Ciudadanía, DAGRD, investigadores, periodistas y el modelo de alerta temprana |
| Riesgo de mal uso | Alto: los 482,93 W/m² nocturnos o el salto de 20,5 m entran a un modelo o a una noticia como si fueran reales | Medio: un usuario ignora las banderas y promedia valores erróneos con válidos | Bajo pero no nulo: tratar la imputación como medición, o usar el producto horario para una alerta que opera a escala de minutos |

**Justificación de la diferencia entre niveles.** Los tres niveles no cambian el fenómeno: cambian la confianza y el contrato sobre el dato. Del crudo al validado el valor no se modifica, pero adquiere juicio: los 482,93 W/m² de las 20:22 en el piranómetro 6004 dejan de ser "un dato" y pasan a ser "un dato erróneo, con bandera, regla y versión de la regla", y los 2.616 minutos ausentes dejan de ser silencio para ser faltante explícito. Del validado al consumo cambian la resolución y las unidades (minuto a hora, pulsos p1/p2 a mm/h), el valor puede imputarse con método documentado y los metadatos crecen hasta volverse contrato de uso: licencia, cita y limitaciones, porque la ciudadanía y el DAGRD no consumen evidencia, consumen producto.

Lo único que nunca cambia es el original: el crudo permanece inmutable como soporte de auditoría, porque en un sistema de alerta temprana la trazabilidad del dato es la trazabilidad de la decisión.

## Pregunta 1.3 — Tres indicadores de calidad para los datos que publica SIATA

### Indicador 1 — Completitud temporal por estación-variable
| Campo | Descripción |
|---|---|
| Nombre y dimensión | Completitud temporal (dimensión: **completitud**, ISO/IEC 25012) |
| Objetivo y pregunta | ¿Qué fracción de los minutos que la estación debió reportar llegaron efectivamente? Detecta silencios de telemetría y huecos de ingesta |
| Fórmula | C(e,v,d) = N_obs(e,v,d) / N_esp(d). N_obs = registros del día d con `calidad` distinto del centinela "faltante"; N_esp = minutos del día según frecuencia nominal (1.440 para 1 min) |
| Unidad y periodicidad | Porcentaje por estación-variable-día; cálculo diario, consolidado mensual |
| Fuente y agregación | Logs de ingesta / CSV publicados; nivel estación → variable → red (promedio ponderado por N_esp) |
| Umbrales y justificación | Aceptable ≥ 99,0 % (≤ 14 min perdidos/día, inocuo para agregados horarios); alerta 95,0–98,9 %; crítico < 95,0 % (> 1 h perdida: los productos horarios y la alerta quedan con huecos invisibles). En el entregable: 35 = 100,00 %, 367 = 99,98 %, 803 = 99,94 % (aceptable) y 6004 = 98,80 % (**alerta**) |
| Responsable y acción | Ingeniero de telemetría / datos de turno: revisar datalogger y enlace, solicitar reenvío de buffer local, abrir incidente; si el crítico supera 24 h, aviso de cobertura en el portal |
| Limitaciones y sesgos | No dice si los valores presentes son buenos (una estación 100 % completa puede estar congelada); si los huecos son regulares (ventana de mantenimiento nocturna) el consolidado mensual los esconde → calcular también por hora del día; depende de que la frecuencia nominal declarada sea correcta |

### Indicador 2 — Proporción de valores válidos según reglas de calidad
| Campo | Descripción |
|---|---|
| Nombre y dimensión | Validez física y coherencia interna (dimensiones: **validez** y **consistencia**) |
| Objetivo y pregunta | ¿Qué fracción de registros sobrevive la cadena de pruebas físicas y de coherencia? Detecta contenido corrupto o imposible antes de que llegue al usuario |
| Fórmula | V(e,v,d) = (N_tot − N_err) / N_tot. N_err = registros con bandera de error en la cadena: (i) rango físico por variable (radiación ≥ 0 y nula con sol bajo horizonte; nivel dentro del rango hidráulico; 0 ≤ HR ≤ 100 %), (ii) salto máximo por minuto, (iii) persistencia congelada ≥ 10 min, (iv) coherencia interna (\|p1 − p2\| ≤ máx(0,5 mm; 20 % del mayor) en la 35) |
| Unidad y periodicidad | Porcentaje por estación-variable-día; cálculo diario |
| Fuente y agregación | Capa validada con banderas versionadas; nivel estación → red |
| Umbrales y justificación | Aceptable ≥ 99,5 %; alerta 98,0–99,4 %; crítico < 98,0 %. Con flujo minutal, 0,5 % ≈ 7 min/día de contenido corrupto es el máximo tolerable antes de sesgar productos hidrológicos; el entregable muestra episodios concentrados (radiación 482,93 W/m² a las 20:22 del 2026-04-30 en 6004; salto 7,6 → 20,5 m en 1 min en 803) que deben aparecer como días críticos aislados |
| Responsable y acción | Líder de calidad de datos: cuarentena de valores con bandera de error, revisión de sensor y calibración, reproceso del día con la versión de regla corregida |
| Limitaciones y sesgos | Depende de la versión de reglas y umbrales (deben publicarse versionados); reglas demasiado estrictas inflan banderas (sesgo pesimista); **no detecta sesgos sin violación de rango**: un sensor descalibrado +2 °C es "válido" pero erróneo |

### Indicador 3 — Latencia de publicación (frescura)
| Campo | Descripción |
|---|---|
| Nombre y dimensión | Latencia de publicación (dimensión: **oportunidad / actualidad**) |
| Objetivo y pregunta | ¿Cuánto tarda un registro, desde que ocurre el fenómeno, en quedar disponible para el usuario con sus banderas? Detecta estaciones ciegas en tiempo operativo |
| Fórmula | L(e,d) = mediana(t_disp − t_fen) sobre registros del día; SLA%(e,d) = fracción de registros con (t_disp − t_fen) ≤ SLA, con SLA = 5 min para consumo de alerta minutal |
| Unidad y periodicidad | Minutos y porcentaje; cálculo por hora, consolidado diario |
| Fuente y agregación | Logs de ingesta y publicación (marca de recepción/publicación vs. `fecha_hora` del fenómeno); nivel estación → red |
| Umbrales y justificación | Aceptable: mediana ≤ 5 min y SLA% ≥ 97 %; alerta: mediana 5–30 min o SLA% 90–97 %; crítico: mediana > 30 min o SLA% < 90 %. Las crecientes súbitas del Valle de Aburrá evolucionan en decenas de minutos: una estación que llega media hora tarde está operacionalmente ciega |
| Responsable y acción | Líder de plataforma/operaciones: revisar cadena de ingesta, activar reenvío desde buffer local, publicar estado de frescura degradada en el portal |
| Limitaciones y sesgos | Exige relojes sincronizados (deriva de reloj entre datalogger y servidor sesga L; usar NTP y documentarlo); no mide contenido; un reproceso puede maquillar la latencia si se cuenta t_disp del reproceso → medir siempre sobre **primera publicación** |

**No redundancia:** los tres miden dimensiones distintas (completitud, validez/consistencia, oportunidad) sobre la misma unidad de análisis (estación-variable-día), son automatizables sobre las columnas `codigo, fecha_hora, valor, calidad`, comparables entre variables y estaciones, y cada uno dispara una acción correctiva con responsable distinto.


## Pregunta 1.4 — Cálculo del Indicador 1 (completitud) sobre (codigo, fecha_hora, valor, calidad)

In [ ]:
import pandas as pd

# La frecuencia nominal por estación se DECLARA desde el catálogo de metadatos
# (WIGOS); nunca se asume. Si faltara, se infiere de la mediana de diferencias
# y se documenta como supuesto.
FREQ_NOMINAL = {"35": "1min", "367": "1min", "6004": "1min", "803": "1min"}
CENT_FALTANTE = {"", "NA", "NaN", "999", "-999"}   # según diccionario de banderas

def completitud_estacion(df: pd.DataFrame, inicio: str, fin: str) -> pd.DataFrame:
    filas = []
    for cod, g in df.groupby("codigo"):
        freq = FREQ_NOMINAL[cod]
        fh   = pd.to_datetime(g["fecha_hora"])
        obs  = fh[~g["calidad"].astype(str).isin(CENT_FALTANTE)]
        per_obs = obs.dt.floor(freq).nunique()          # períodos con dato válido
        per_esp = len(pd.date_range(inicio, fin, freq=freq, inclusive="left"))
        filas.append({"codigo": cod, "frecuencia_nominal": freq,
                      "periodos_esperados": per_esp,
                      "periodos_observados": per_obs,
                      "completitud": per_obs / per_esp})
    return pd.DataFrame(filas)

# Nivel red, comparable entre estaciones con distinto intervalo:
# rejilla horaria común; una hora está cubierta si tiene >= 1 observación válida.
def horas_cubiertas_red(df: pd.DataFrame) -> pd.Series:
    g = df[~df["calidad"].astype(str).isin(CENT_FALTANTE)].copy()
    g["hora"] = pd.to_datetime(g["fecha_hora"]).dt.floor("h")
    return g.groupby(["codigo", "hora"]).size().groupby(level=0).size()

Tratamiento de estaciones con intervalos de muestreo distintos:

- Rejilla propia por estación: la completitud se calcula contra la frecuencia nominal de cada estación (declarada en metadatos), nunca contra una rejilla ajena: comparar conteos crudos entre una estación de 1 min y una de 10 min sería comparar tamaños, no calidades.

- Comparabilidad de red: para agregar o comparar entre estaciones se normaliza a una rejilla común (horaria): "hora cubierta" = al menos una observación válida dentro de la hora; ahí el indicador sí es comparable sin importar el intervalo original.
Agregación documentada: al llevar una serie de alta frecuencia a la rejilla común se declara la regla por tipo de variable (lluvia = suma del período; nivel = promedio o valor de fin de período), jamás un remuestreo implícito.

- Intervalo variable en el tiempo: si un datalogger fue reconfigurado a mitad del período, se segmenta la serie por punto de cambio y se calcula completitud por segmento con su propia frecuencia, dejando el cambio en el linaje.
Zona horaria: todos los cálculos en UTC o en la zona declarada con su offset (America/Bogotá, UTC−5), para no fabricar huecos o duplicados de 60 minutos en cambios de hora.


## Pregunta 1.5 — Práctica concreta por grupo de estándares

| Estándar | Tema | Aplicación concreta en SIATA |
|---|---|---|
| DAMA-DMBOK 2 | Gobierno y dimensiones de calidad | Nombrar un propietario y un steward por variable (precipitación, nivel, radiación, meteorología) y gestionar la calidad por dimensiones —completitud, validez, oportunidad, consistencia— con un indicador medible y un responsable por dimensión, tal como el marco de gobierno de DMBOK lo prescribe. |
| ISO 8000 e ISO/IEC 25012 y 25024 | Modelo y medición de calidad del dato | Adoptar las características de 25012 como modelo y las métricas de 25024 como método de medición, y publicar una declaración de calidad por conjunto: por ejemplo "piranómetro 6004: completitud 98,80 % y validez 99,2 % en dic-2025 a abr-2026", para que la calidad sea un dato medido y no un adjetivo. |
| ISO 19115 y 19157 | Metadatos y calidad geoespacial | Documentar cada estación con metadatos geoespaciales (coordenadas WGS84, altura del sensor, exposición, cuenca) y los elementos de calidad de 19157 (linaje, exactitud temática y posicional) en el catálogo del portal, para que cualquier usuario evalúe aptitud de uso sin preguntar a SIATA. |
| OMM No. 8 y No. 100 | Calidad de instrumentos y de datos climatológicos | Aplicar calibración con trazabilidad a patrones referenciados OMM con periodicidad definida por tipo de sensor, y procedimientos de control climatológico (rangos por clima local, p. ej. radiación global limitada por el máximo de cielo despejo) antes de publicar cualquier valor climatológico. |
| OMM No. 1131 y metadatos WIGOS | Gestión de calidad y metadatos de estaciones | Registrar cada estación en WIGOS con identificador único y metadatos completos (ubicación, exposición, instrumentación, historial de mantenimiento y calibración) y operar el control de calidad según la guía OMM No. 1131 con banderas trazables por prueba aplicada. |
| Principios FAIR | Acceso y reutilización de datos | Publicar con identificador persistente (DOI), licencia abierta legible por máquina (CC BY), API documentada y metadatos estructurados, de modo que un tercero pueda encontrar, acceder, interoperar y reutilizar los datos sin mediación de SIATA. |

## Pregunta 1.6 — Cadena de control de calidad en cinco etapas

Diccionario de banderas usado: 0 = válido, 1 = faltante, 2 = cuestionable (revisión), 3 = erróneo (no usar), 4 = corregido/estimado. Ninguna bandera sobrescribe el valor original.

| Etapa | Prueba concreta | Umbral | Bandera |
|---|---|---|---|
| 1. Formato y completitud | Parseo de cada fila contra el esquema (nº de campos, timestamp ISO, tipos numéricos) y contraste contra la rejilla esperada de 1.440 min/día por estación | Fila con número de campos distinto o timestamp no parseable (p. ej. el registro fusionado `…0.254,151235,2025-12-01 18:10…` de la estación 35) → erróneo; minuto ausente respecto a la rejilla → faltante | 3 / 1 |
| 2. Rango físico | Rangos por variable: radiación 0–1.300 W/m² y = 0 con sol bajo el horizonte; humedad 0–100 %; nivel dentro del rango hidráulico de la estación (−2 a 15 m en 803); precipitación por minuto ≤ 2 mm | Radiación 482,93 W/m² a las 20:22 del 2026-04-30 (noche) o nivel 20,5 m en 803 fuera de rango → erróneo | 3 |
| 3. Coherencia temporal y persistencia | Salto máximo por minuto (nivel \|Δ\| ≤ 0,5 m/min; radiación \|Δ\| ≤ 150 W/m²/min) y valor congelado (idéntico ≥ 10 min consecutivos en variable que debe variar) | Salto 7,6 → 20,5 m en 1 min en 803 (2026-03-16 00:03) → erróneo; meseta de 18,10 m durante 14 min (2026-03-17 06:59–07:12) → cuestionable | 3 / 2 |
| 4. Coherencia interna entre variables | En 35: \|p1 − p2\| ≤ máx(0,254 mm; 20 % del mayor); en 367: punto de rocío ≤ temperatura y coherencia humedad–temperatura; radiación vs. elevación solar | \|p1 − p2\| = 0,254 con p2 = 0 repetido en minutos secos → cuestionable | 2 |
| 5. Coherencia espacial | Comparación con la red: lluvia aislada en 35 sin eco en 367 (misma cuenca); radiación de 6004 contra modelo de cielo despejo y contra piranómetros vecinos | Desviación > 3 sigmas robustas de la red en hora húmeda, o evento aislado con red seca → cuestionable | 2 |

Las banderas 2 y 3 alimentan una cola de revisión humana; solo tras revisión un valor puede pasar a 4 (corregido/estimado) con método documentado.



## Pregunta 1.7 — Arquitectura híbrida de datos para SIATA

### (a) Conceptos base y elección
- **Modelos de servicio:** IaaS = infraestructura alquilada (máquinas virtuales, redes); PaaS = plataforma gestionada (servicios de ingesta, cómputo y bases sin administrar servidores); SaaS = software consumido tal cual (dashboards comerciales).
- **Modelos de despliegue:** pública (recursos multi-inquilino del proveedor), privada (infraestructura exclusiva de la entidad), híbrida (combinación orquestada de ambas).
- **Elección para SIATA: híbrida con predominio PaaS en nube pública (AWS).** La escala elástica de ingesta minutal y la publicación abierta son nativas de la nube; pero la alerta temprana es misión crítica y no puede depender del enlace a internet, por lo que el núcleo de alerta y el buffer permanecen on-premise/edge.

### (b) Capas de almacenamiento y formatos
- **Crudo:** object storage (S3) con los archivos tal como llegan, inmutables (Object Lock / WORM), particionado `raw/año/mes/día/codigo/`; retención permanente.
- **Validado:** Parquet comprimido (zstd) particionado por fecha y estación, con columnas de banderas y `version_reglas`; es la capa de trabajo y auditoría.
- **Producto:** agregados horarios/diarios en Parquet + tablas de servicio para API y dashboards; formatos columnares y particionamiento garantizan consultas rápidas y costos bajos.

### (c) Orquestación, cómputo y reprocesamiento
Orquestación con Airflow gestionado (MWAA): un DAG por frecuencia (minutal de ingesta, horario de validación, diario de productos). Cómputo: funciones serverless (Lambda) para ingesta y contenedores (Fargate/Glue) para validación y agregación. **Idempotencia:** cada escritura es un upsert con clave (codigo, fecha_hora) más `run_id` de linaje; los reprocesos sobrescriben particiones completas de forma determinista con las reglas versionadas, de modo que re-ejecutar nunca duplica ni depende del orden.

### (d) Tabla del modelo híbrido

| Componente | On-premise | Nube | Justificación |
|---|---|---|---|
| Adquisición y buffer local en estación o centro de datos | Sí (principal) | No | El datalogger/gateway con buffer store-and-forward garantiza que no se pierda ningún minuto ante cortes de enlace o del proveedor |
| Datos crudos históricos | Copia local acotada (30–90 días) | Sí (S3, fuente de verdad) | Durabilidad y costo por TB favorecen a la nube; lo local solo cubre la ventana operativa crítica |
| Validación y control de calidad | Sí (reglas ligeras en tiempo real) | Sí (reproceso completo) | La alerta no puede esperar a la nube; el reproceso completo con reglas versionadas y pesadas vive en la nube |
| Repositorio de calidad (catálogo, banderas, reglas) | Caché de lectura | Sí (fuente de verdad) | Catálogo único centralizado con réplica local de lectura para operar sin enlace |
| Publicación para usuarios (API, datos abiertos) | Espejo degradado de contingencia | Sí (API + CDN + open data) | Escala global, cache y costo de publicación son nativos de la nube |
| Modelos y alertas de operación crítica | Sí (primario) | Sí (respaldo y reentrenamiento) | La alerta de la ciudad debe funcionar sin internet; la nube da redundancia, reentrenamiento y análisis retrospectivo |

**Sincronización y continuidad:** replicación asíncrona edge→nube con cola store-and-forward (ningún minuto se descarta por falta de enlace); replicación cruzada de capas validadas; si cae el enlace, el edge sigue alertando con reglas locales y buffer; si cae el proveedor, el sitio on-premise sirve publicación degradada y alerta primaria. Metas: RPO ≤ 5 min y RTO ≤ 30 min para alerta; RTO ≤ 4 h para publicación.

### (e) Repositorio de calidad en la nube
Catálogo (Glue Data Catalog u OpenMetadata) con diccionario de variables y banderas publicado desde el repo Git; reglas de validación como código (pruebas dbt / Great Expectations) con versionado semántico; linaje automático (OpenLineage) desde ingesta hasta producto; indicadores de calidad (completitud, validez, latencia) en tablero operativo; salidas por API REST (FastAPI) y datos abiertos en bucket público/CKAN. Control de acceso por rol con IAM (público solo-lectura de productos; analistas lectura de validado; ingenieros escritura en crudo/validado); retención declarada por capa; respaldo con versionado + replicación cruzada; DR con RPO/RTO de (d).

### (f) Seguridad y cumplimiento
Identidades federadas (SSO) y roles IAM con mínimo privilegio y recertificación periódica; cifrado en reposo (KMS) y en tránsito (TLS 1.3); secretos en gestor dedicado, nunca en código; normativa colombiana: Ley 1581 de 2012 (protección de datos personales — aplica a bitácoras de mantenimiento y contactos, no al dato ambiental), Ley 1712 de 2014 (transparencia y acceso a información pública, que obliga a publicar proactivamente), y guías MinTIC/ISO 27001 para el SGSI.

### (g) Observabilidad
Monitoreo continuo de los tres indicadores de calidad por estación con semáforo y alerta automática ante silencio > 15 min (completitud), caída de validez o latencia fuera de SLA; métricas del pipeline (lag de ingesta, reintentos, fallas de DAG) con dead-man switch; tableros Grafana/CloudWatch y guardia turnada.

### (h) Costos y tres decisiones de control
Cualitativo: almacenamiento barato, cómputo moderado, **egreso (salida de datos) es el costo dominante** por descargas abiertas de series minutales. Decisiones: (1) servir descargas vía CDN con cache y formatos comprimidos/columnares (Parquet/zip) para reducir egress; (2) políticas de lifecycle que muevan crudo > 24 meses a almacenamiento de archivo; (3) publicar agregados horarios/diarios al público general y reservar el minutal a API con cuotas y registro, más cómputo de reproceso en instancias spot/autoescaladas.

### (i) Infraestructura como código y despliegue continuo
Terraform en módulos por capa (ingesta, storage, cómputo, publicación, seguridad) con estados remotos y revisión de planes en PR; CI/CD con GitHub Actions: pruebas → `terraform plan` → aprobación → `apply` → despliegue de código de validación; detección de drift y rollback por versión.

```mermaid
flowchart LR
  E[Estaciones: datalogger + buffer edge] --> G[Gateway on-prem: reglas ligeras tiempo real]
  G --> A[Motor de alertas local - primario]
  G -->|store-and-forward TLS| K[AWS: API GW + Lambda ingesta]
  K --> Q[S3 crudo WORM particionado]
  Q --> V[Validacion Fargate/Glue - reglas versionadas]
  V --> W[S3 validado Parquet + banderas]
  W --> P[Productos: agregados horarios/diarios]
  V --> C[Repositorio de calidad: catalogo, diccionario, reglas semver, linaje, indicadores]
  P --> API[API FastAPI + CDN + datos abiertos]
  C --> API
  W -->|replicacion| L[(Espejo local critico 30-90 dias)]
  L --> A
```

## Pregunta 1.8 — Flujo de trabajo Git para equipo de cinco personas

- **Estrategia de ramas:** trunk-based: `main` protegida equivale a producción; ramas `feature/*` y `fix/*` de vida corta (≤ 2 días) salen de `main` y vuelven por pull request; `hotfix/*` para correcciones urgentes con despliegue inmediato; releases mediante etiquetas, sin ramas largas tipo develop que divergen.
- **Convención de commits:** Conventional Commits con ámbito: `feat(qc-radiacion): agrega prueba de radiacion nocturna`, `fix(carga): tolera filas fusionadas del CSV`, `docs(diccionario): documenta bandera 2`; un commit = un cambio lógico; mensaje en imperativo y referencia al issue.
- **Revisión de código:** PR obligatorio con al menos un aprobador y CODEOWNERS sobre `reglas/` y `diccionario/`; checklist de revisión: prueba nueva o actualizada, evidencia con fixture, diccionario y CHANGELOG al día, sin secretos.
- **Pruebas automáticas (CI):** GitHub Actions en cada PR: lint (ruff), pruebas unitarias de cada regla de calidad con fixtures sintéticos que incluyen anomalías conocidas (pico nocturno, meseta congelada, fila fusionada), prueba de integración sobre una muestra de los CSV y construcción de docs; merge bloqueado si algo falla.
- **Gestión de versiones del algoritmo de calidad:** versionado semántico por etiquetas (v1.4.0); cada regla con identificador y versión propia (p. ej. RNG-RAD-01 v3); las salidas graban `version_conjunto_reglas` en cada fila validada para auditoría y reproceso; CHANGELOG.md con cambios rupturistas en major.
- **Manejo de datos grandes y secretos:** los crudos nunca entran a Git (viven en S3/DVC con checksums); el repo contiene solo fixtures sintéticos (< 1 MB) y salidas doradas para regresión; secretos fuera del repo mediante variables de entorno y gestor de secretos, con gitleaks en pre-commit y en CI.
- **README de calidad:** propósito y alcance; diagrama de arquitectura; requisitos e instalación con versiones fijadas; cómo ejecutar todo con un comando; fuentes de datos y licencia; diccionario de variables, reglas y banderas; cómo correr las pruebas; esquema de versionado y changelog; guía de contribución (ramas, PR, commits); propietarios y contacto; limitaciones conocidas.

### Pregunta 1.9 — Auditoría del portal de datos SIATA (https://datos.siata.gov.co/)

### (a) ¿Qué cambiaría? Elementos existentes que deben modificarse

A1. Granularidad excesiva: un dataset por estación en lugar de uno por red
Evidencia: el listado muestra entradas independientes para "6001 - Piranómetro Torre SIATA", "6002 - Piranómetro Concejo Itagüí", "6003", "6004", "6005"… y así por cada una de las ~150 estaciones. El dataset "Información de la Red de Piranómetro" existe como conjunto separado, no como metadato embebido.

Principio: ISO 19115 § B.2.4 (granularidad apropiada del conjunto de datos) y principio Findable de FAIR: una red hidrometeorológica es conceptualmente un conjunto de datos con dimensión espacial, no N conjuntos atómicos.

Cambio: consolidar cada red (pluviométrica, piranómetro, nivel, meteorológica) en un dataset multivariado con dimensión estación, y dejar los datasets por estación solo como vista descargable. El dataset de metadatos de red se convierte en la tabla maestra de estaciones dentro del dataset de red.

A2. Descripciones de datasets reducidas a una frase
Evidencia: la descripción visible en el listado es literalmente "Histórico de la estación 6004 - Piranómetro Parque de las Aguas" — no incluye variables, resolución, unidades, rango temporal, tamaño, formato ni fecha de última actualización. El investigador debe abrir cada uno de los 495 datasets para saber si le sirve.
Principio: ISO 19115 (resumen descriptivo obligatorio) y principio Findable de FAIR (los metadatos deben permitir el descubrimiento sin abrir el recurso).
Cambio: enriquecer el resumen con campos estructurados: variables, resolución temporal, rango, formato, tamaño, DOI de referencia, fecha de actualización.

### (b) ¿Qué falta? Capacidades, metadatos, documentación o controles ausentes**

B1. Ausencia de API programática visible
Evidencia: en la página principal del portal no aparece enlace a API REST, endpoint de consulta ni librería cliente (Python/R). Las 364.959 descargas sugieren consumo humano vía descarga de ficheros, no integración con sistemas aguas abajo.
Principio: principio Accessible de FAIR ("mediante un protocolo de comunicación estandarizado, gratuito y abierto") y Ley 1712 de 2014 (transparencia activa con acceso programático).
Falta: API documentada con ejemplos de código, consulta por estación/variable/rango, paginación y formato JSON/Parquet.

B2. Sin indicadores de calidad ni estado operativo
Evidencia: no se publica por dataset ni por estación ningún indicador de completitud, proporción de valores válidos, latencia ni estado operativo (en línea / caída / en mantenimiento). Un desarrollador que consuma los datos no sabe si la estación 803 está reportando hoy.
Principio: ISO/IEC 25024 (medición de calidad del dato) y OMM No. 1131 (monitoreo continuo de calidad).
Falta: panel público de salud de la red con los tres indicadores propuestos en la Pregunta 1.3 y estado operativo en tiempo real.

B3. Sin documentación de uso ni ejemplos reproducibles
Evidencia: no se encuentra sección "Cómo usar estos datos", notebook de ejemplo, ni plantilla de análisis. El ciudadano o el investigador novato descarga un CSV de 40 MB sin guía de interpretación.
Principio: principio Reusable de FAIR y práctica de datos abiertos (Open Data Charter).
Falta: tutoriales reproducibles (Jupyter/Rmarkdown) por tipo de análisis: series de tiempo, anomalías, correlación espacial, pronóstico.

### (c) ¿Qué sobra? Redundancias, información confusa o procesos que no agregan valor

C1. Redundancia estructural: datasets por estación + dataset de red separados
Evidencia: coexisten N datasets atómicos por estación (ej: "6001 Piranómetro Torre SIATA" … "6005 Piranómetro AMVA") y un dataset "Información de la Red de Piranómetro". El metadato maestro vive aparte del dato, obligando al usuario a reconciliar dos fuentes.
Principio: DAMA-DMBOK 2 (normalización y eliminación de duplicados).
Redundancia: el dataset de red debería ser la tabla maestra embebida en el dataset consolidado de la red.

C2. Jerarquía Dataverse / Dataset poco clara para el ciudadano
Evidencia: el portal expone simultáneamente "Dataverses (10)" y "Datasets (485)" sin explicar al usuario no técnico la diferencia ni cuál usar. Un ciudadano que quiera "la lluvia de Medellín" no sabe si buscar en Dataverses o Datasets.
Principio: usabilidad y transparencia (Ley 1712 de 2014).
Redundancia: el ciudadano debería ver una vista simplificada ("Temas" o "Variables") que oculte la taxonomía interna del repositorio.